# W86 / #44 — GPU/TPU Substrate with Deterministic Replay (Colab Pro)

Closes the P2 #44 line on a real GPU. Runs two arms against an open-weight transformer under the W80 instrumentation contract:

1. **Positive arm (deterministic ON, default)** — replay-from-KV byte-identity at the bf16 GPU floor; two consecutive forwards produce identical trace CIDs; hidden-state intercept moves the trace CID.
2. **Negative arm (deterministic OFF)** — `torch.use_deterministic_algorithms(False)` + `cudnn.benchmark=True` — replay diff jumps OR consecutive forwards differ by CID. Proves the wrapper is load-bearing.

## One-time setup (do this before *Run all*)

1. **Runtime → Change runtime type → A100 GPU** (L4 / V100 also fine; T4 borderline at 8B).
2. **Set HF token Colab Secret** (🔑 left sidebar → *+ Add new secret* → name=`hf_token`, value=`hf_xxxxxxxx`, toggle *Notebook access* on).
3. *Runtime → Run all*. Total wall: ~6 min on A100 (~2 × 90 s model load + ~90 s benches).

## Output

* `gpu_substrate_v1_bench_report.json` at `/content/w86_gpu_<TS>/` + Drive backup at `MyDrive/coordpy_gpu/w86_gpu_<TS>/`.
* Per-DoD-bullet PASS/FAIL summary from `verify_w86_gpu_substrate_v1_audit_chain.py`.

## Anti-cheat

* Both arms run **the same** model + tokenizer + prompt.
* Negative arm only flips the determinism wrapper.
* Replay-from-KV tolerance is the **W86 contract's bf16 tier** (`W86_REPLAY_TOLERANCE_PER_TIER`).
* All CIDs re-derive offline.

In [ ]:
# --- 1. Environment probe + workdir ---
import os, sys, subprocess, json, datetime, shutil
RUN_TS = datetime.datetime.now(
    tz=datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = f'/content/w86_gpu_{RUN_TS}'
os.makedirs(OUT_DIR, exist_ok=True)
print('RUN_TS:', RUN_TS)
print('OUT_DIR:', OUT_DIR)
subprocess.run(['nvidia-smi'], check=False)
import torch
print('torch:', torch.__version__,
      'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  '
          f'mem={p.total_memory / (1024**3):.2f} GiB  '
          f'sm={p.major}.{p.minor}')

In [ ]:
# --- 2. Install transformers + accelerate ---
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0',
    'accelerate>=0.34.0',
    'huggingface_hub>=0.24.0',
    'numpy>=1.24',
    'cryptography>=41.0',
], check=True)
import transformers, accelerate, huggingface_hub
print('transformers:', transformers.__version__)
print('accelerate:', accelerate.__version__)
print('huggingface_hub:', huggingface_hub.__version__)

In [ ]:
# --- 3. HF login via Colab Secrets ---
from google.colab import userdata  # type: ignore
hf_token = userdata.get('hf_token').strip()
assert hf_token.startswith('hf_'), (
    'Colab Secret `hf_token` must start with hf_. '
    'Add it via the key icon in the left sidebar.')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)
print('HF login OK; token len =', len(hf_token))

In [ ]:
# --- 4. Clone CoordPy + install in editable mode ---
shutil.rmtree('/content/CoordPy', ignore_errors=True)
subprocess.run([
    'git', 'clone', '--depth=1',
    'https://github.com/adotdong29/CoordPy.git',
    '/content/CoordPy',
], check=True)
os.chdir('/content/CoordPy')
print('HEAD =', subprocess.check_output(
    ['git', 'log', '-1', '--oneline'], text=True).strip())
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
], check=True)
import coordpy
print('coordpy:', coordpy.__version__,
      'SDK:', getattr(coordpy, 'SDK_VERSION', '?'))

In [ ]:
# --- 5. Run the GPU substrate driver ---
GPU_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
GPU_LOG = os.path.join(OUT_DIR, 'gpu_run.log')
print('GPU_MODEL:', GPU_MODEL)
print('GPU_LOG:  ', GPU_LOG)
!cd /content/CoordPy && python scripts/run_w86_gpu_substrate_v1_bench.py \
    --out-dir {OUT_DIR} \
    --model-name {GPU_MODEL} \
    --device cuda:0 \
    --precision-tier tier_bf16 \
    --prompt-tokens 32 \
    --inject-layer 4 \
    --inject-magnitude 1.0 2>&1 | tee {GPU_LOG}
print()
print('=== OUT_DIR after driver ===')
!ls -la {OUT_DIR}

In [ ]:
# --- 6. Offline re-verifier (per-DoD-bullet PASS/FAIL) ---
RPT = os.path.join(
    OUT_DIR, 'gpu_substrate_v1_bench_report.json')
!cd /content/CoordPy && python scripts/verify_w86_gpu_substrate_v1_audit_chain.py \
    --report {RPT}

In [ ]:
# --- 7. Print headline numbers ---
with open(RPT) as fh:
    overall = json.load(fh)
rep = overall.get('report', {})
print('=== W86 #44 GPU Substrate V1 closure headline ===')
for k in ('model_name', 'device', 'precision_tier',
          'cuda_device_name', 'cuda_capability',
          'tier_tolerance',
          'pos_replay_max_abs_diff',
          'pos_replay_within_tier_tolerance',
          'pos_intercept_moves_cid',
          'pos_forwards_byte_identical',
          'neg_replay_max_abs_diff',
          'neg_replay_breaks_byte_identity',
          'tp_readback_passthrough_byte_identical',
          'report_cid'):
    if k in rep:
        print(f'  {k}: {rep[k]}')
print()
print('=== arm wall times ===')
print(f"  pos_arm_wall_seconds = {overall.get('pos_arm_wall_seconds')}")
print(f"  neg_arm_wall_seconds = {overall.get('neg_arm_wall_seconds')}")

In [ ]:
# --- 8. Save to Drive + zip download ---
from google.colab import drive  # type: ignore
drive.mount('/content/drive', force_remount=False)
DEST = (
    f'/content/drive/MyDrive/coordpy_gpu/'
    f'w86_gpu_{RUN_TS}')
os.makedirs(DEST, exist_ok=True)
subprocess.run(
    f'cp -r {OUT_DIR}/. {DEST}/', shell=True, check=False)
print('saved to Drive:', DEST)
subprocess.run(f'ls -la {DEST}', shell=True, check=False)
zip_path = f'/content/w86_gpu_{RUN_TS}.zip'
subprocess.run(
    ['zip', '-rq', zip_path, f'w86_gpu_{RUN_TS}'],
    cwd='/content', check=False)
print('zip ready at', zip_path,
      f'({os.path.getsize(zip_path)/1024:.1f} KiB)')
from google.colab import files  # type: ignore
files.download(zip_path)